# Polynomial-cost orbital optimization of cluster number quasisymetries

In [27]:
# Molecule input parameters
molecule = 'h2o'        # Supports: h2, h2o, n2, lih, h4_linear, h4_square
bond_length = 1.1      # Angstrom
bond_dim = 5         # DMRG bond dimension
basis = 'sto3g'       # Basis set

# DMRG parameters
max_bond_dim = 5
n_sweeps = 2

# High-level input parameters
num_clusters = 3
initial_basis = 'MOs' # 'MOs': HF molecular orbitals, #TODO: 'NOs': natural orbitals, 'localized': orbitals localized on nuclei via...
custom_cluster_matrix = None # None for automated search; #TODO: (num_clusters - 1) x norb with chemically-motivated clusters
type_cost_function = 'variance' # 'variance' or #TODO: 'eval_eq'
num_transfers = 2 # maximal number of inter-cluster electron transfers starting from fundamental sector

In [28]:
# import new source code

import sys
sys.path.insert(0, '..')
import src.cluster_number_operators as cluster_ops

In [29]:
import ffsim
print("=== 1. TESTING FFSIM ===")
try:
    # Set up a tiny 2-orbital, 2-electron (1 alpha, 1 beta) space
    norb, nelec = 2, (1, 1)
    
    # Create a simple Hartree-Fock state (first orbital occupied)
    state = ffsim.hartree_fock_state(norb, nelec)
    
    # Define a simple 2x2 orbital rotation matrix (Givens rotation / beam splitter)
    theta = np.pi / 4
    U = np.array([[np.cos(theta), -np.sin(theta)], 
                  [np.sin(theta),  np.cos(theta)]])
    
    # Apply the orbital rotation to the state
    rotated_state = ffsim.apply_orbital_rotation(state, U, norb, nelec)
    
    print(f"ffsim OK! State vector shape: {rotated_state.shape}")
except Exception as e:
    print(f"ffsim failed! Error: {e}")

import os
import shutil
from pyblock2.driver.core import DMRGDriver, SymmetryTypes

print("=== VERIFYING BLOCK2 C++ BACKEND ===")
try:
    scratch_dir = "./tmp_scratch"
    
    # 1. Initialize driver
    driver = DMRGDriver(scratch=scratch_dir, symm_type=SymmetryTypes.SZ, n_threads=1)
    
    # 2. Set up a 4-orbital, 4-electron space (spin 0 = singlet)
    driver.initialize_system(n_sites=4, n_elec=4, spin=0)
    
    # 3. Ask the C++ backend to generate a random MPS wavefunction tensor
    ket = driver.get_random_mps(tag="TEST_MPS", bond_dim=10, nroots=1)
    
    print("Success! block2 is 100% working.")
    print(f"Generated an MPS wavefunction with {ket.n_sites} sites.")
    
    # Clean up
    if os.path.exists(scratch_dir):
        shutil.rmtree(scratch_dir)
        
except Exception as e:
    print(f"block2 failed! Error: {e}")

=== 1. TESTING FFSIM ===
ffsim OK! State vector shape: (4,)
=== VERIFYING BLOCK2 C++ BACKEND ===
Success! block2 is 100% working.
Generated an MPS wavefunction with 4 sites.


In [30]:
# ============================================================================
# SECTION 1: Get psi_mps with DMRG using MOs as sites; get 1- and 2-RDMs
# ============================================================================

import numpy as np
import pyscf
from chemistry import get_geometry_and_description
from dmrg_solver import Block2DMRGSolver, DMRGConfig, solve_or_load_ground_state

# --- Step 1.1: Build molecule and run HF ---
geometry, _ = get_geometry_and_description(molecule, bond_length)
mol = pyscf.M(atom=geometry, basis=basis)
mf = pyscf.scf.RHF(mol)
mf.kernel()

norb, nelec = mol.nao, mol.nelec
h1e = mf.mo_coeff.T @ mf.get_hcore() @ mf.mo_coeff
g2e = pyscf.ao2mo.full(mol, mf.mo_coeff)
ecore = mol.energy_nuc()

# --- Step 1.2: Run DMRG ---
import tempfile
store_dir = tempfile.mkdtemp(prefix='dmrg_')
solver = Block2DMRGSolver(
    h1e=h1e, g2e=g2e, ecore=ecore,
    n_elec=nelec, spin=0,
    store_dir=store_dir, n_threads=1
)
result = solve_or_load_ground_state(
    solver,
    config=DMRGConfig(max_bond_dim=max_bond_dim, n_sweeps=n_sweeps),
    reuse=False
)
print(f"DMRG Energy: {result.energy:.10f} Ha")

# --- Step 1.3: Extract RDMs directly from MPS ---
solver._activate()
mps = solver.get_mps()

rdm1 = solver.driver.get_1pdm(mps)
rdm2 = solver.driver.get_2pdm(mps) # .transpose(0, 3, 1, 2) to get chemistry notation

print("Type of rdm1:", type(rdm1))
print("rdm1:\n", rdm1)

converged SCF energy = -74.9418025664523
DMRG Energy: -75.0053006588 Ha
Type of rdm1: <class 'list'>
rdm1:
 [array([[ 9.99999112e-01, -2.51510551e-05,  3.83815769e-06,
         1.03356247e-07,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [-2.51510551e-05,  9.97301496e-01,  1.81858155e-04,
        -3.32619086e-03,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 3.83815769e-06,  1.81858155e-04,  9.78304823e-01,
         2.45472346e-04,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 1.03356247e-07, -3.32619086e-03,  2.45472346e-04,
         9.84781530e-01,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  9.99298373e-01,  0.00000000e+00,
         0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  2.21470909e-02,
        -7.94397416e-05],
       [ 0.00000000e+00,  0

In [31]:
# ============================================================================
# SECTION 2: Get best clusters, described via a binary cluster_matrix
# (if custom_cluster_matrix = None). Should use Praveen's beam search.
# ============================================================================


In [32]:
# ============================================================================
# SECTION 3: Get the optimal orbital basis, starting from initial_basis.
# Optimization cost function type is given by type_cost_function.
# ============================================================================

In [33]:
# ============================================================================
# SECTION 4: Get labels of dominant sector and sectors obtained from it by 
# moving up to num_transfers electrons
# ============================================================================

In [34]:
# ============================================================================
# SECTION 5: For small systems, evaluate quality of sector decomposition
# by plotting K_sectors -> ΔE, K_states -> ΔE
# ============================================================================